<p align="center">\n<a href="https://silvimetric.com"><img src="https://github.com/hobuinc/silvimetric/blob/main/docs/source/logo/Logos/PNG/SilviMeteric_Logo_2c.png?raw=true" width="50%"></a></p>\n\n# SilviMetric Rainy Lake Demo\n\nThis Colab notebook reads an existing SilviMetric TileDB database from S3 and extracts forestry metric rasters from it. The default database is `s3://rainy-lake-sivilmetric-31b-test/db`.\n

### Workflow\n\n1. Install the Colab conda runtime and SilviMetric dependencies.\n2. Configure S3 access for TileDB.\n3. Open the existing Rainy Lake SilviMetric database.\n4. Inspect database metadata.\n5. Extract selected metrics to GeoTIFF rasters.\n

# Installations

## Install CondaColab\n\nCondaColab installs a conda-compatible runtime in Colab. After this cell finishes, Colab usually restarts the runtime. Run the next cell after the restart completes.\n

In [ ]:
%pip install git+https://github.com/hobu/condacolab.git@py312\n\nimport condacolab\ncondacolab.install_miniforge()\n

## Check CondaColab Install

In [ ]:
import time\ntime.sleep(10)\n\nimport condacolab\ncondacolab.check()\n

## Install SilviMetric Dependencies\n\nSilviMetric uses PDAL, GDAL, and TileDB from conda-forge. This cell restarts the runtime after installing packages so Python sees the newly installed environment.\n

In [ ]:
%%capture\n%env PROJ_NETWORK=TRUE\n%env PROJ_DATA=/usr/local/condacolab/share/proj\n\n!mamba update --all --yes\n!mamba install silvimetric rasterio geopandas folium mapclassify openssl sqlite pystac --yes\n\nprint("Restarting session.")\nimport os\nos.kill(os.getpid(), 9)\n

In [ ]:
%env PROJ_NETWORK=TRUE\n%env PROJ_DATA=/usr/local/condacolab/share/proj\n\n!pdal --version\n!gdalinfo --version\n!silvimetric --version\n!mamba list tiledb\n!mamba list openssl\n!mamba list sqlite\n

## Setup

This notebook reads a public S3-backed TileDB database. It configures TileDB directly for unsigned public reads in `us-west-2`, so no AWS credentials or `AWS_*` environment variables are required.


In [ ]:
import os
import json
from pathlib import Path

import tiledb
import rasterio
from rasterio.plot import show
from rasterio.windows import get_data_window, transform
import silvimetric as sm
from silvimetric.resources.metrics.stats import statistics

os.environ.setdefault("SILVIMETRIC_TILEDB_S3_REGION", "us-west-2")
os.environ.setdefault("SILVIMETRIC_TILEDB_NO_SIGN_REQUEST", "true")
os.environ.setdefault("SILVIMETRIC_TILEDB_MAX_INCOMPLETE_RETRIES", "1000")

tiledb_config = tiledb.Config()
tiledb_config["vfs.s3.region"] = os.environ["SILVIMETRIC_TILEDB_S3_REGION"]
tiledb_config["vfs.s3.no_sign_request"] = os.environ["SILVIMETRIC_TILEDB_NO_SIGN_REQUEST"]
tiledb_config["vfs.s3.connect_scale_factor"] = "25"
tiledb_config["vfs.s3.connect_max_retries"] = "10"
tdb_ctx = tiledb.Ctx(tiledb_config)

log = sm.Log(10)
db_path = "s3://rainy-lake-sivilmetric-31b-test/db"
out_dir = Path.cwd() / "rasters"
out_dir.mkdir(exist_ok=True)

db_path, out_dir.as_posix()


## Open the Rainy Lake Database\n\nThe database is already initialized and populated, so this notebook skips the original initialize, scan, shatter, delete, resume, and restart cells. Those operations mutate a database; here we only inspect and extract from the S3 TileDB store.\n

In [ ]:
try:
    object_type = tiledb.object_type(db_path, ctx=tdb_ctx)
except tiledb.TileDBError as error:
    raise RuntimeError(
        f"Could not open TileDB database at {db_path}. "
        "Check that the S3 bucket/key exists and is publicly readable."
    ) from error

print(f"TileDB object type: {object_type}")

if object_type == "group":
    with tiledb.Group(db_path, "r", ctx=tdb_ctx) as group:
        shard_members = [(member.name, member.uri) for member in group]
    first_shard_uri = shard_members[0][1]
    print(f"Shard count: {len(shard_members)}")
    print(f"First shard: {first_shard_uri}")
else:
    shard_members = []
    first_shard_uri = db_path

# Avoid sm.info(db_path) here: group info opens every shard, which is
# expensive over public S3. Use the first shard for a quick schema check.
quick_info = sm.info(first_shard_uri, concise=True)
print(json.dumps(quick_info, indent=2, default=str)[:5000])


In [ ]:
storage = sm.Storage.from_db(first_shard_uri, ctx=tdb_ctx)

print("Root bounds:", storage.config.root)
print("CRS:", storage.config.crs)
print("Resolution:", storage.config.resolution)
print("Attributes:", [attr.name for attr in storage.config.attrs])
print("Metrics:", [metric.name for metric in storage.config.metrics])


## Extract Metric Rasters

By default, this extracts a single `Intensity` / `max` raster from the first public shard so the demo completes quickly. The full Rainy Lake group has hundreds of shards; extracting the whole group is intentionally left as an optional cell.


In [ ]:
import datetime

available_attr_names = {attr.name for attr in storage.config.attrs}\navailable_metric_names = {metric.name for metric in storage.config.metrics}\n\nextract_attrs = [sm.Attributes["Intensity"]] if "Intensity" in available_attr_names else None\nextract_metrics = [statistics["max"]] if "max" in available_metric_names else None\n\nextract_config = sm.ExtractConfig(\n    tdb_dir=first_shard_uri,\n    log=log,\n    out_dir=out_dir.as_posix(),\n    attrs=extract_attrs,\n    metrics=extract_metrics,\n)\n\nsm.extract(extract_config)\nsorted(path.name for path in out_dir.glob("*.tif"))\n

In [ ]:
raster_paths = sorted(out_dir.glob("*.tif"))\nraster_path = raster_paths[0]\n\nwith rasterio.open(raster_path) as raster:\n    profile = raster.profile.copy()\n    data_window = get_data_window(raster.read(masked=True))\n    data_transform = transform(data_window, raster.transform)\n    profile.update(\n        transform=data_transform,\n        height=data_window.height,\n        width=data_window.width,\n    )\n    print(raster_path)\n    print(profile)\n    show(raster.read(window=data_window))\n

## Optional: Extract the Full Database\n\nThe following cell is intentionally commented out. Uncomment it only when you want every stored attribute/metric combination written to GeoTIFF.\n

In [ ]:
import datetime

# full_extract_config = sm.ExtractConfig(\n#     tdb_dir=db_path,\n#     log=log,\n#     out_dir=(Path.cwd() / "all_rasters").as_posix(),\n#     attrs=None,\n#     metrics=None,\n# )\n# sm.extract(full_extract_config)\n